# LC 261 — Graph Valid Tree
**Difficulty:** Medium | **Pattern:** Union-Find / Cycle Detection

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> A valid tree has exactly two properties:
exactly <code>n-1</code> edges (no extra), and no cycles (all nodes
connected without redundant paths). Check edge count first — it's
a free early exit. Then use Union-Find: if two nodes in an edge
already share a root, adding that edge creates a cycle.
</div>

## Official Problem Statement

Given `n` nodes labeled `0` to `n-1` and a list of undirected edges,
determine if these edges form a valid tree.

A valid tree must be:
- Connected (every node reachable from every other node)
- Acyclic (no cycles)

**Constraints:**
- `1 <= n <= 2000`
- `0 <= edges.length <= 5000`
- `edges[i].length == 2`
- `0 <= ai, bi < n`
- `ai != bi`
- No repeated edges

## What This Is Actually Asking

A tree is a graph with **no cycles** and **exactly one component**.

The mathematical shortcut: a connected acyclic graph on `n` nodes
**always** has exactly `n-1` edges. So:

- Too few edges → graph is disconnected (forest, not tree)
- Too many edges → graph has at least one cycle
- Exactly `n-1` edges AND no cycle → valid tree

This means we only need to verify:
1. `len(edges) == n - 1`
2. No cycle exists (using Union-Find)

## Walk Through an Example by Hand

```
Valid tree:  n=5, edges=[[0,1],[0,2],[0,3],[1,4]]
  len(edges)=4 == n-1=4 ✓
  parent = [0,1,2,3,4]
  (0,1): find(0)=0, find(1)=1 → diff → union → parent=[0,0,2,3,4]
  (0,2): find(0)=0, find(2)=2 → diff → union → parent=[0,0,0,3,4]
  (0,3): find(0)=0, find(3)=3 → diff → union → parent=[0,0,0,0,4]
  (1,4): find(1)=0, find(4)=4 → diff → union → parent=[0,0,0,0,0]
  No cycle found → return True ✓

Invalid (cycle):  n=5, edges=[[0,1],[1,2],[2,3],[1,3],[1,4]]
  len(edges)=5 != n-1=4 → return False immediately ✓

Invalid (cycle, right edge count):  n=4, edges=[[0,1],[1,2],[2,3],[0,3]]
  len(edges)=4 != n-1=3 → return False immediately ✓
```

## The Picture

```
Valid Tree (n=5):        Invalid (cycle, n=4):

    0                       0 — 1
   /|\                      |   |
  1 2 3                     3 — 2
  |                         ^
  4                    cycle! 0-1-2-3-0

Union-Find detects cycle:
  When processing edge (u,v):
    find(u) == find(v)?  → SAME root → cycle exists!
    find(u) != find(v)?  → different roots → safe, union them

parent array evolution (valid tree):
  start:  [0, 1, 2, 3, 4]
  +(0,1): [0, 0, 2, 3, 4]
  +(0,2): [0, 0, 0, 3, 4]
  +(0,3): [0, 0, 0, 0, 4]
  +(1,4): [0, 0, 0, 0, 0]  ← single root = connected!
```

## When To Use This Pattern

Use Union-Find for "valid tree" / cycle detection when:
- Processing edges incrementally
- Need to detect if adding an edge creates a cycle
- Checking if a graph is a spanning tree
- Kruskal's MST algorithm (same core logic)

**The two-condition shortcut** (`n-1` edges + no cycle) works because
these conditions together imply connectivity. You never need to
explicitly verify connectivity as a third check.

**Watch out:** `n=1, edges=[]` is a valid tree (single node).

## The Approach

1. **Early exit:** if `len(edges) != n - 1`, return `False`
2. Initialize `parent[i] = i` for all `i` in `0..n-1`
3. Define `find(x)` with path compression:
   - `if parent[x] != x: parent[x] = find(parent[x])`
   - `return parent[x]`
4. For each edge `(u, v)`:
   - `ru, rv = find(u), find(v)`
   - If `ru == rv`: **cycle detected** → return `False`
   - Else: `parent[ru] = rv` (union)
5. Return `True` (n-1 edges, no cycle → valid tree)

**Time:** O(n + E·α(n)) ≈ O(n)  
**Space:** O(n)

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """Run test cases for validTree."""
    tests = [
        # (n, edges, expected)
        (5, [[0,1],[0,2],[0,3],[1,4]],      True),
        (5, [[0,1],[1,2],[2,3],[1,3],[1,4]], False),
        (1, [],                              True),
        (2, [[0,1]],                         True),
        (2, [],                              False),
        (4, [[0,1],[2,3]],                   False),
        (4, [[0,1],[1,2],[2,3],[0,3]],       False),
        (3, [[0,1],[1,2]],                   True),
    ]
    passed = 0
    for n, edges, expected in tests:
        result = func(n, edges)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: n={n}, edges={edges}"
            )
            print(
                f"    expected={expected}, got={result}"
            )
    total = len(tests)
    print(f"\nResults: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")

In [ ]:
def validTree(n: int, edges: List[List[int]]) -> bool:
    """
    Determine if edges form a valid tree using Union-Find.

    Args:
        n: number of nodes labeled 0..n-1
        edges: list of undirected edges [u, v]

    Returns:
        True if the graph is a valid tree, False otherwise.

    Strategy:
        - A valid tree needs exactly n-1 edges AND no cycles
        - Check len(edges)==n-1 first (free early exit)
        - Use Union-Find: if find(u)==find(v) for any edge,
          that edge creates a cycle → return False
        - If no cycle found after all edges → return True

    Example:
        n=5, edges=[[0,1],[0,2],[0,3],[1,4]] → True
        n=5, edges=[[0,1],[1,2],[2,3],[1,3],[1,4]] → False
    """
    # Debug: print inputs
    print(f"  n={n}, edges={edges}")
    print(f"  len(edges)={len(edges)}, n-1={n-1}")

    pass

    # Debug: print parent array after each union
    # print(f"  parent={parent}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(validTree)

## Complexity

| | Value | Why |
|---|---|---|
| **Time** | O(n + E·α(n)) | Init n nodes + process E edges with near-O(1) UF ops |
| **Space** | O(n) | Parent array of size n |

The `len(edges) != n-1` early exit is O(1) and catches many
invalid cases before any Union-Find work is needed.

**Alternative approaches:**
- DFS/BFS: Track visited nodes; if you revisit a node (other than
  the parent), a cycle exists. Then check all nodes were visited.
  O(n + E) time and space.
- Union-Find is cleaner here: no adjacency list needed, processes
  the edge list directly.

## Real World Connection

**File systems:** A directory tree must be acyclic and connected —
exactly the valid tree constraint. Symlinks that create cycles
break filesystem traversal for the same reason.

**Network topology:** A spanning tree of a network provides
loop-free routing. Protocols like STP (Spanning Tree Protocol)
use this exact logic to detect and eliminate cycles that would
cause broadcast storms.

**Version control:** Git's commit DAG (directed acyclic graph)
is a generalization — each branch history must not form cycles
to ensure a consistent history.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra